<a href="https://colab.research.google.com/github/otiedemann/sdc-tobe/blob/main/drone_detection/scripts/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLO11n Drone Detection — Training Notebook

**Runtime:** Go to `Runtime → Change runtime type → T4 GPU`

**Time:** ~2-3 hours for full training on T4

**Result:** `best.onnx` model file (~5 MB) to download

## 1. Check GPU and install dependencies

In [4]:
# Verify GPU is available
!nvidia-smi
print("\n" + "="*50)
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Mon Apr 13 19:45:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Install ultralytics (YOLO framework) and huggingface_hub (dataset download)
!pip install -q ultralytics huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.8 MB/s eta 0:00:00


## 2. Download Seraphim Drone Detection Dataset

83,483 labeled drone images (640×640), YOLO format, ~9 GB.

This takes **10-20 minutes** on Colab's network.

In [6]:
from huggingface_hub import snapshot_download
import os

DATA_DIR = "/content/seraphim"

if not os.path.exists(f"{DATA_DIR}/train/images"):
    print("Downloading Seraphim dataset (~9 GB)...")
    print("This takes 10-20 minutes. Go grab a coffee.\n")
    snapshot_download(
        repo_id="lgrzybowski/seraphim-drone-detection-dataset",
        repo_type="dataset",
        local_dir=DATA_DIR,
        ignore_patterns=["*.md", ".gitattributes"],
    )
    print("\nDownload complete!")
else:
    print("Dataset already downloaded.")

# Verify
for split in ["train", "test"]:
    img_dir = f"{DATA_DIR}/{split}/images"
    lbl_dir = f"{DATA_DIR}/{split}/labels"
    n_img = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f"  {split}: {n_img} images, {n_lbl} labels")

This takes 10-20 minutes. Go grab a coffee.



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]


Download complete!
  train: 4 images, 1 labels
  test: 1 images, 1 labels


## 3. Create dataset config

In [7]:
# Write dataset YAML
yaml_content = f"""path: {DATA_DIR}
train: train/images
val: test/images

names:
  0: drone
"""

with open("/content/drone_dataset.yaml", "w") as f:
    f.write(yaml_content)

print("Dataset YAML:")
print(yaml_content)

Dataset YAML:
path: /content/seraphim
train: train/images
val: test/images

names:
  0: drone



## 4. Train YOLO11n

Fine-tunes YOLO11n (pretrained on COCO) on drone detection.

- **Epochs:** 100 with early stopping (patience=25)
- **Time:** ~2-3 hours on T4
- **Auto batch size** to fit GPU memory

In [ ]:
from ultralytics import YOLO

# Load YOLO11n pretrained on COCO
model = YOLO("yolo11n.pt")

# Train
results = model.train(
    data="/content/drone_dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=-1,              # auto batch size (fills ~60% VRAM)
    device=0,              # GPU
    patience=25,           # early stopping
    workers=2,             # Colab has limited CPU cores
    project="/content/runs",
    name="drone_yolo11n",
    exist_ok=True,
    # Augmentation
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    # Training params
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    warmup_epochs=3.0,
    # Logging
    verbose=True,
    plots=True,
)

print("\n" + "="*50)
print("Training complete!")
print(f"Best model: /content/runs/drone_yolo11n/weights/best.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 5. View training results

In [ ]:
from IPython.display import Image, display
import os

results_dir = "/content/runs/drone_yolo11n"

# Show training curves
for plot in ["results.png", "confusion_matrix.png", "P_curve.png", "R_curve.png"]:
    path = f"{results_dir}/{plot}"
    if os.path.exists(path):
        print(f"\n{plot}:")
        display(Image(filename=path, width=800))

In [ ]:
# Show validation predictions
val_path = f"{results_dir}/val_batch0_pred.jpg"
if os.path.exists(val_path):
    print("Validation batch predictions:")
    display(Image(filename=val_path, width=800))

## 6. Evaluate on test set

In [ ]:
# Load best model and run validation
best_model = YOLO("/content/runs/drone_yolo11n/weights/best.pt")
metrics = best_model.val(data="/content/drone_dataset.yaml", device=0)

print(f"\n{'='*50}")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

## 7. Export to ONNX (for deployment on RPi / flight controller)

The ONNX model works with OpenCV DNN — no PyTorch needed on the Pi.

In [ ]:
# Export to ONNX
best_model = YOLO("/content/runs/drone_yolo11n/weights/best.pt")
onnx_path = best_model.export(format="onnx", imgsz=640, simplify=True, opset=12)
print(f"\nONNX model exported: {onnx_path}")

# Check file size
import os
size_mb = os.path.getsize(onnx_path) / 1e6
print(f"File size: {size_mb:.1f} MB")

## 8. Quick inference test

In [ ]:
import cv2
import glob
import numpy as np
from IPython.display import Image as IPImage, display
import time

# Test on a few images from the test set
test_images = sorted(glob.glob(f"{DATA_DIR}/test/images/*.jpg"))[:5]

model = YOLO("/content/runs/drone_yolo11n/weights/best.pt")

for img_path in test_images:
    t0 = time.monotonic()
    results = model(img_path, conf=0.4, imgsz=640, verbose=False)
    elapsed = (time.monotonic() - t0) * 1000

    # Save annotated image
    annotated = results[0].plot()
    out_path = f"/content/test_result_{os.path.basename(img_path)}"
    cv2.imwrite(out_path, annotated)

    n_dets = len(results[0].boxes)
    print(f"{os.path.basename(img_path)}: {n_dets} drone(s) detected ({elapsed:.0f}ms)")
    display(IPImage(filename=out_path, width=640))

## 9. Download the trained model

Download these files to your computer, then copy `best.onnx` to your flight controller Pi.

In [ ]:
from google.colab import files

print("Downloading best.onnx (ONNX — for deployment on Pi)...")
files.download("/content/runs/drone_yolo11n/weights/best.onnx")

In [ ]:
# Also download the .pt file (in case you want to fine-tune further later)
print("Downloading best.pt (PyTorch — for further training)...")
files.download("/content/runs/drone_yolo11n/weights/best.pt")

## 10. (Optional) Save to Google Drive

In case the Colab session times out, save your model to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = "/content/drive/MyDrive/drone_detection_model"
os.makedirs(save_dir, exist_ok=True)

for f in ["best.pt", "best.onnx", "last.pt"]:
    src = f"/content/runs/drone_yolo11n/weights/{f}"
    if os.path.exists(src):
        shutil.copy2(src, save_dir)
        print(f"Saved {f} → {save_dir}/")

# Also save training plots
for f in ["results.png", "confusion_matrix.png"]:
    src = f"/content/runs/drone_yolo11n/{f}"
    if os.path.exists(src):
        shutil.copy2(src, save_dir)

print(f"\nAll saved to Google Drive: {save_dir}")

---

## Deployment

Copy `best.onnx` to your flight controller Pi, then use it:

```bash
# Copy model to Pi
scp best.onnx pi@flightctrl1:~/sdc-tobe/drone_detection/models/

# Test on live stream
python drone_detection/drone_detector.py \
    --api http://localhost:8080 \
    --model drone_detection/models/best.onnx
```

The `DroneDetector` class uses only OpenCV DNN — no PyTorch needed on the Pi.